# 🎭 Avatar Renderer — **Premium Engines on Colab GPU**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruslanmv/avatar-renderer-mcp/blob/main/demo_colab_premium.ipynb)

This is the **second** Colab notebook — it runs the *premium* lip-sync engines that
**cannot** run on the Hugging Face **ZeroGPU** Space:

| Engine | What it is | Free **T4** (16 GB) |
|---|---|:--:|
| **MuseTalk** | Real-time latent-space lip-sync (MIT) | ✅ |
| **Diff2Lip** | Diffusion lip-sync | ✅ |
| **Wav2Lip** | GAN lip-sync (research license) | ✅ |
| **LatentSync** | Diffusion lip-sync, highest quality (OpenRAIL++) | ⚠️ **Pro/A100 only** (T4 OOMs) |

> **Why these don't run on ZeroGPU:** ZeroGPU only gives the GPU to in-process code
> inside `@spaces.GPU`. These engines run as **subprocesses** that need their own
> repos + multi-GB weights, so they can't see the ZeroGPU allocation. A real,
> persistent Colab GPU runs them natively.

**Runtime:** `Runtime → Change runtime type → T4 GPU` before you start.

## 1 · Check the GPU

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No GPU — set Runtime → Change runtime type → T4 GPU')

import torch
assert torch.cuda.is_available(), 'No CUDA GPU. Enable it via Runtime → Change runtime type → T4 GPU, then re-run.'
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {name}  |  VRAM: {vram_gb:.1f} GB  |  torch {torch.__version__}')

# T4 (~16 GB): MuseTalk + Diff2Lip + Wav2Lip. LatentSync needs more headroom.
LATENTSYNC_OK = vram_gb >= 22.0
print('LatentSync enabled:' , LATENTSYNC_OK, '(needs an A100/L4 — Colab Pro)')

## 2 · Clone the project & install Python deps

Installs the repo (which gives us the engine registry + `orchestrate()`),
ffmpeg, and the lip-sync deps. We do **not** touch Colab's pre-installed CUDA torch.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path('/content/avatar-renderer-mcp')
if not REPO_DIR.exists():
    !git clone -q https://github.com/ruslanmv/avatar-renderer-mcp.git {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('repo at', REPO_DIR)

# System ffmpeg (video mux).
!apt-get -qq install -y ffmpeg > /dev/null

# Lip-sync runtime deps (kept off Colab's torch). librosa pin matches Wav2Lip audio.
!pip install -q 'librosa>=0.10' soundfile opencv-python-headless imageio imageio-ffmpeg \
    huggingface_hub 'gradio_client' edge-tts gfpgan basicsr facexlib numpy scipy 2>/dev/null || true
print('deps installed')

## 3 · Set the engine environment

`orchestrate()` finds engines via these paths. Setting them now means the
availability probes in `app.engines` light up once the repos + weights are present.

In [ ]:
MODEL_ROOT = str(REPO_DIR / 'models')
EXT_DEPS   = str(REPO_DIR / 'external_deps')
os.environ['MODEL_ROOT']   = MODEL_ROOT
os.environ['EXT_DEPS_DIR'] = EXT_DEPS
os.environ['LIPSYNC_HF_REPO'] = 'ruslanmv/avatar-renderer'
# Put every engine repo on the path (so their inference scripts import).
for sub in ['first-order-model','Wav2Lip','SadTalker','Diff2Lip','guided-diffusion','MuseTalk','LatentSync']:
    p = f'{EXT_DEPS}/{sub}'
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ['PYTHONPATH'] = ':'.join([str(REPO_DIR)] + [f'{EXT_DEPS}/{s}' for s in
    ['first-order-model','Wav2Lip','SadTalker','Diff2Lip','guided-diffusion','MuseTalk','LatentSync']])
print('MODEL_ROOT =', MODEL_ROOT)
print('EXT_DEPS   =', EXT_DEPS)

## 4 · Download the core weights

Pulls FOMM (motion), Wav2Lip, Diff2Lip and GFPGAN from the project's model repo
`ruslanmv/avatar-renderer` (the same checkpoints `hf/start.sh` uses).

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

CORE = [
    ('fomm/vox-cpk.pth',        'fomm/vox-cpk.pth'),
    ('wav2lip/wav2lip_gan.pth', 'wav2lip/wav2lip_gan.pth'),
    ('diff2lip/Diff2Lip.pth',   'diff2lip/Diff2Lip.pth'),
    ('gfpgan/GFPGANv1.3.pth',   'gfpgan/GFPGANv1.3.pth'),
]
for remote, local in CORE:
    dst = Path(MODEL_ROOT) / local
    if dst.exists():
        print('[OK]', local); continue
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        p = hf_hub_download(repo_id='ruslanmv/avatar-renderer', filename=remote, repo_type='model')
        shutil.copy2(p, dst)
        print('[DL]', local, f'{dst.stat().st_size/1e6:.0f} MB')
    except Exception as e:
        print('[SKIP]', local, str(e)[:120])

## 5 · Install the premium engine repos + weights

Uses the repo's own installer, then fetches MuseTalk's weights. On the **free T4**
we set up **MuseTalk + Diff2Lip**. LatentSync is only attempted on a high-VRAM GPU.

In [ ]:
# Clone engine repos (FOMM/Diff2Lip/guided-diffusion/MuseTalk) via the repo's script.
!bash scripts/download_enhancements.sh musetalk diff2lip 2>&1 | tail -25

# Make sure FOMM is present (needed by every pipeline engine for the motion stage).
fomm = Path(EXT_DEPS) / 'first-order-model'
if not (fomm / '.git').exists():
    !git clone -q --depth 1 https://github.com/AliaksandrSiarohin/first-order-model.git {fomm}
print('FOMM present:', fomm.exists())

In [ ]:
# MuseTalk weights → models/musetalk/ (checkpoint + configs). First inference may
# also pull auxiliary weights (sd-vae, whisper, dwpose) into the MuseTalk repo.
muse_dir = Path(MODEL_ROOT) / 'musetalk'
muse_dir.mkdir(parents=True, exist_ok=True)
if not any(muse_dir.glob('*.bin')) and not any(muse_dir.glob('*.safetensors')):
    !huggingface-cli download TMElyralab/MuseTalk --local-dir {muse_dir} 2>&1 | tail -8
# Run MuseTalk's official weight downloader if it ships one (gets the auxiliaries).
dw = Path(EXT_DEPS) / 'MuseTalk' / 'download_weights.sh'
if dw.exists():
    !cd {EXT_DEPS}/MuseTalk && bash download_weights.sh 2>&1 | tail -8
print('MuseTalk weights dir:', list(muse_dir.glob('*'))[:6])

In [ ]:
# OPTIONAL — LatentSync (only on A100/L4 / Colab Pro; skipped on T4).
if LATENTSYNC_OK:
    !bash scripts/download_enhancements.sh latentsync 2>&1 | tail -15
    ls_dir = Path(MODEL_ROOT) / 'latentsync'; ls_dir.mkdir(parents=True, exist_ok=True)
    !huggingface-cli download chunyu-li/LatentSync --local-dir {ls_dir} 2>&1 | tail -8
    print('LatentSync installed')
else:
    print('Skipping LatentSync — not enough VRAM on this GPU (need A100/L4).')

## 6 · Which engines are ready?

The registry probes the repos + weights we just installed and reports availability,
GPU need, and **commercial-license** status — the same logic the production
orchestrator uses.

In [ ]:
# Re-import so the env vars set above take effect.
import importlib, app.engines, app.compliance
importlib.reload(app.compliance); importlib.reload(app.engines)
from app.engines import registry

rows = registry.info_all()
print(f"{'engine':14} {'avail':6} {'gpu':4} {'commercial':11} license")
print('-'*70)
for r in rows:
    print(f"{r['name']:14} {str(r['available']):6} {str(r['requires_gpu']):4} "
          f"{str(r['commercial_ok']):11} {r['license']}")
available = [r['name'] for r in rows if r['available']]
print('\nAVAILABLE:', available)

## 7 · Generate — pick your engine

Type text (synthesized with edge-tts) **or** upload audio, then choose the engine.
If the engine you pick isn't available it falls back to the best one that is.

In [ ]:
#@title Render settings { run: "auto" }
engine = "musetalk"  #@param ["musetalk", "diff2lip", "wav2lip", "latentsync", "wav2lip_fast", "simple"]
quality_mode = "high_quality"  #@param ["standard", "high_quality", "premium"]
text = "Hello! This avatar is running a premium lip-sync engine on a Colab GPU."  #@param {type:"string"}
voice = "en-US-AriaNeural"  #@param ["en-US-AriaNeural","en-US-GuyNeural","en-GB-SoniaNeural","es-ES-ElviraNeural","fr-FR-DeniseNeural"]
print('engine =', engine, '| quality =', quality_mode)

In [ ]:
# Provide a portrait. Upload your own, or this uses the bundled example if present.
from google.colab import files
import glob

print('Upload a PORTRAIT image (or press Cancel to use a bundled sample):')
up = files.upload()
if up:
    IMAGE = str(Path('/content') / list(up.keys())[0])
    Path(IMAGE).write_bytes(list(up.values())[0])
else:
    cand = glob.glob(str(REPO_DIR / 'zerogpu/assets/*.png')) + glob.glob(str(REPO_DIR / 'assets/*.png'))
    IMAGE = cand[0] if cand else None
print('portrait:', IMAGE)

In [ ]:
# Audio: synthesize from `text`, or upload your own WAV/MP3.
import asyncio, edge_tts, uuid

print('Upload AUDIO to override the text (or Cancel to use text-to-speech):')
ua = files.upload()
if ua:
    AUDIO = str(Path('/content') / list(ua.keys())[0])
    Path(AUDIO).write_bytes(list(ua.values())[0])
else:
    AUDIO = f'/content/tts_{uuid.uuid4().hex[:8]}.mp3'
    async def _go():
        await edge_tts.Communicate(text, voice).save(AUDIO)
    asyncio.get_event_loop().run_until_complete(_go())
print('audio:', AUDIO)

In [ ]:
# Run the multi-engine orchestrator. Honors availability + license; on a non-strict
# tier it softly falls back to the best available engine if the pick isn't ready.
from app.render import orchestrate

req = engine
if req not in available:
    fallback = next((e for e in ['musetalk','diff2lip','wav2lip','wav2lip_fast','simple'] if e in available), 'simple')
    print(f'⚠️  {req!r} not available here — using {fallback!r} instead.')
    req = fallback

OUT = '/content/avatar_premium.mp4'
out = orchestrate(face_image=IMAGE, audio=AUDIO, out_path=OUT,
                  quality_mode=quality_mode, engine=req)
print('rendered:', out)

## 8 · Watch & download

In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(Path(out).read_bytes()).decode()
HTML(f'<video width=480 controls autoplay loop src="data:video/mp4;base64,{data}"></video>')

In [ ]:
from google.colab import files
files.download(out)

---
### Notes & troubleshooting
- **CUDA out of memory** on T4 → use `engine="musetalk"` or `"diff2lip"`, lower the
  portrait resolution, and avoid `latentsync` (needs A100/L4).
- **MuseTalk first run** may download auxiliary weights (sd-vae, whisper, dwpose);
  this is one-time per session. Mount Drive (`MODEL_ROOT` on Drive) to persist them.
- **An engine shows `available: False`** → re-run §5 for that engine and re-run §6;
  the registry re-probes the repo + weights.
- This notebook uses `orchestrate()` — the **same** production path as the API, so
  what works here is what ships. Commercial-use flags are shown in §6.